#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("schemaBronze", "bronze")
dbutils.widgets.text("catalogo", "PROJECT_DEV")
dbutils.widgets.text("storageName", "adlssmartprojectdev13")

In [0]:
container = dbutils.widgets.get("container")
schemaBronze = dbutils.widgets.get("schemaBronze")
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

# definir rutas
path_inicio = f"abfss://{container}@{storageName}.dfs.core.windows.net/raw_data_project/"
path_target = f"{catalogo}.{schemaBronze}."

In [0]:
# importar functions essentials
from pyspark.sql.functions import *
from pyspark.sql.types import *

## PRODUCTS

In [0]:
products_schema = StructType(fields=[StructField("product_id", StringType(), False),
                                     StructField("product_category_name", StringType(), True),
                                     StructField("product_name_length", IntegerType(), True),
                                     StructField("product_description_length", IntegerType(), True),
                                     StructField("product_photos_qty", IntegerType(), True),
                                     StructField("product_weight_g", IntegerType(), True),
                                     StructField("product_length_cm", IntegerType(), True),
                                     StructField("product_height_cm", IntegerType(), True),
                                     StructField("product_width_cm", IntegerType(), True)
])

In [0]:
df_products = spark.read.format("csv") \
    .option("header", True) \
    .schema(products_schema) \
    .load(f"{path_inicio}olist_products_dataset.csv")
#.schema(orders_items_schema) \

## PRODUCT CATEGORY

In [0]:
orders_product_category = StructType(fields=[StructField("product_category_name", StringType(), False),
                                     StructField("product_category_name_english", StringType(), True)
])

In [0]:
df_product_category = spark.read.format("csv") \
    .option("header", True) \
    .schema(orders_product_category) \
    .load(f"{path_inicio}product_category_name_translation.csv")

## cargar informacion tablas target

In [0]:
df_products.write.mode("overwrite").insertInto(f"{path_target}products")

In [0]:
df_product_category.write.mode("overwrite").insertInto(f"{path_target}products_category")

## aplicar optimize y vacuum

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${esquema_target}.products
ZORDER BY product_id;

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${esquema_target}.products RETAIN 24 HOURS ;

VACUUM ${catalogo}.${esquema_target}.products_category RETAIN 24 HOURS;